# Cox Proportional Hazard Modeling 2: Gene Expression 
Now, we will do univariate testing of individual gene expression to see which effect survival the most

In [21]:
import pandas as pd
import numpy as np
from lifelines import CoxPHFitter
import pyhere as here

In [3]:
here.here()

PosixPath('/Users/jmakings/Documents/Projects/breast_cancer_survival_prediction')

### Preprocessing Functions

In [4]:
def preprocess_for_cox(df, duration_col, event_col):
    # Drop rows with missing values in duration or event columns
    df = df.dropna(subset=[duration_col, event_col])
    
    # Ensure duration column is numeric
    df[duration_col] = pd.to_numeric(df[duration_col], errors='coerce')
    
    # Drop rows with non-numeric duration values
    df = df.dropna(subset=[duration_col])
    
    return df

In [5]:
# converts float columns with whole numbers to integer type
def convert_whole_float_columns_to_int(df):
    for col in df.columns:
        if pd.api.types.is_float_dtype(df[col]):
            # Check if all non-null values are whole numbers
            if df[col].dropna().apply(float.is_integer).all():
                df[col] = df[col].astype("Int64")  # nullable integer type
    return df

In [6]:
def one_hot_encode(df, convert_whole_num_ints=True):
    # One hot encode categorical variables

    # Identify categorical columns automatically
    cat_cols = df.select_dtypes(include=["object", "category"]).columns

    # Apply one-hot encoding only to categorical columns
    df_encoded = pd.get_dummies(df, columns=cat_cols, drop_first=True)

    # Change boolean columns to integers
    bool_cols = df_encoded.select_dtypes(include=["bool"]).columns
    df_encoded[bool_cols] = df_encoded[bool_cols].astype(int)
    
    if convert_whole_num_ints:
        df_encoded = convert_whole_float_columns_to_int(df_encoded)
    return df_encoded

### Load data and Preprocess

In [7]:
# load clinical metadata with PCA + UMAP features
pca_clinical_df = pd.read_csv(here.here("data", "processed","clinical_dimred_features.csv"))

In [8]:
# load expression data
expression_data_df = pd.read_csv(here.here("data", "processed","expression_data.csv"))

In [9]:
# One hot encode categorical variables

df_encoded = one_hot_encode(pca_clinical_df)
df_encoded = convert_whole_float_columns_to_int(df_encoded)
# Rename for clarity
df_encoded = df_encoded.rename(columns={
    "overall_survival_months": "time",
    "death_from_cancer_Living": "event"
})

In [16]:
# gene expression data with time and cancer survival columns
survival_exp = pd.concat([df_encoded[["time","event"]], expression_data_df], axis=1)

# Ensure no time = 0 for proportional hazards testing
EPS = 1e-4  # or smaller if time is in months
survival_exp['time_adj'] = survival_exp['time'].clip(lower=EPS)

#### Time = 0 will cause runtime error when log-transforming, so we clip it to a very small number

In [18]:
(survival_exp['time']==0).sum()

1

In [17]:
(survival_exp['time_adj']==0).sum()

0

## Screen Genes Individually for Cox Modeling

We will look at each gene individually first, observing which genes violate our proportional hazard assumption

In [19]:
from lifelines import CoxPHFitter
from lifelines.statistics import proportional_hazard_test

ph_results = []

for gene in expression_data_df.columns:
    print(f"Processing gene: {gene}")
    df_tmp = survival_exp[['time_adj', 'event', gene]].dropna()

    # Skip genes with near-zero variance
    if df_tmp[gene].std() < 1e-6:
        continue

    cph = CoxPHFitter()
    cph.fit(df_tmp, 'time_adj', 'event')

    # Perform proportional hazards test on log time to model early vs late effects better
    ph_test = proportional_hazard_test(cph, df_tmp, time_transform='log')

    ph_results.append({
        'gene': gene,
        'ph_p': ph_test.summary.loc[gene, 'p'],
        'coef': cph.params_[gene],
        'hr': cph.hazard_ratios_[gene],
    })

ph_df = pd.DataFrame(ph_results)
ph_df['ph_violation'] = ph_df['ph_p'] < 0.01



Processing gene: brca1
Processing gene: brca2
Processing gene: palb2
Processing gene: pten
Processing gene: tp53
Processing gene: atm
Processing gene: cdh1
Processing gene: chek2
Processing gene: nbn
Processing gene: nf1
Processing gene: stk11
Processing gene: bard1
Processing gene: mlh1
Processing gene: msh2
Processing gene: msh6
Processing gene: pms2
Processing gene: epcam
Processing gene: rad51c
Processing gene: rad51d
Processing gene: rad50
Processing gene: rb1
Processing gene: rbl1
Processing gene: rbl2
Processing gene: ccna1
Processing gene: ccnb1
Processing gene: cdk1
Processing gene: ccne1
Processing gene: cdk2
Processing gene: cdc25a
Processing gene: ccnd1
Processing gene: cdk4
Processing gene: cdk6
Processing gene: ccnd2
Processing gene: cdkn2a
Processing gene: cdkn2b
Processing gene: myc
Processing gene: cdkn1a
Processing gene: cdkn1b
Processing gene: e2f1
Processing gene: e2f2
Processing gene: e2f3
Processing gene: e2f4
Processing gene: e2f5
Processing gene: e2f6
Processing

### Create time interactions for genes violating porportional hazard

In [23]:
# create interaction terms for genes with proportional hazards violations
violating_genes = ph_df.loc[ph_df.ph_violation, 'gene'].tolist()

df_model = survival_exp.copy()

for gene in violating_genes:
    df_model[f'{gene}_logt'] = df_model[gene] * np.log(df_model['time_adj'])


In [ ]:
# 23 genes with proportional hazards violations
len(violating_genes)

23

In [24]:
df_model

,time,event,brca1,brca2,palb2,pten,tp53,atm,cdh1,chek2,...,pdgfa_logt,ptk2_logt,asxl1_logt,asxl2_logt,fancd2_logt,sf3b1_logt,smarcc1_logt,tbl1xr1_logt,ubr5_logt,rdh5_logt
0,140.500000,1,-1.3990,-0.5738,-1.6217,1.4524,0.3504,1.1517,0.0348,0.1266,...,5.113345,-7.865847,3.155537,-7.026645,-10.373067,0.050936,-1.329272,-1.182894,-7.941509,24.779940
1,84.633333,1,-1.3800,0.2777,-1.2154,0.5296,-0.0136,-0.2659,1.3594,0.7961,...,-1.529448,-3.395765,-1.104256,3.436598,-3.155208,11.189025,-5.952242,-0.407439,2.267542,-2.812569
2,163.700000,0,0.0670,-0.8426,0.2114,-0.3326,0.5141,-0.0803,1.1398,0.4187,...,-7.105132,-5.400859,-9.624581,-11.698972,-3.140900,-3.358586,1.260234,-2.098861,2.514861,-3.851056
3,164.933333,1,0.6744,-0.5428,-1.6592,0.6369,1.6708,-0.8880,1.2491,-1.1889,...,-2.131564,13.421958,-8.654403,-9.022002,-1.472949,-3.163904,-3.526397,-2.263286,8.204094,-6.384479
4,41.366667,0,1.2932,-0.9039,-0.7219,0.2168,0.3484,0.3897,0.9131,0.9356,...,-4.481116,11.506916,-2.855883,-3.802136,-2.167970,3.872119,3.291041,-1.244796,11.544141,4.828423
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1899,196.866667,1,0.1563,0.5543,-0.6149,0.4572,1.3822,-0.0537,-0.1323,0.2837,...,-0.983606,-6.882076,-4.802345,-4.372347,-1.297389,7.771653,0.864750,0.867391,-2.859960,-3.422021
1900,44.733333,0,0.1343,0.9128,1.3017,0.7383,0.1841,0.0028,0.1243,2.2040,...,-5.982712,-0.966903,-8.173066,0.521079,10.754514,-1.010611,-2.370128,-1.218510,1.063821,-2.386091
1901,175.966667,0,1.8107,-0.2608,0.4006,-0.2985,0.0356,-0.1620,1.5486,1.5309,...,-7.355778,-11.999220,-10.570667,-5.262326,0.414141,5.470689,3.881340,0.783817,4.545206,-8.460670
1902,86.233333,0,-1.2746,-1.7695,-0.3454,-0.3850,0.6689,1.4531,1.0956,-0.0948,...,-3.617347,-2.729502,-8.075296,-1.265804,-3.454219,7.010059,-1.525651,6.115528,-4.444131,-3.804544


### Create new model with all genes, including time covariates for genes violating PH

In [ ]:
# create interaction terms for genes with proportional hazards violations
violating_genes = ph_df.loc[ph_df.ph_violation, 'gene'].tolist()

df_model = survival_exp.copy()

for gene in violating_genes:
    df_model[f'{gene}_logt'] = df_model[gene] * np.log(df_model['time_adj'])

# Define model variables
base_genes = ph_df.loc[
    (ph_df.ph_violation == False) & (ph_df.ph_p > 0.05),
    'gene'
].tolist()

# genes without proportional hazards violations
len(base_genes)

435

In [ ]:
# create covariates list, combining base genes, violating genes, and their interaction terms
covariates = (
    base_genes +
    violating_genes +
    [f'{g}_logt' for g in violating_genes]
)

In [ ]:
# fit new model with interaction terms
# penalizer and l1_ratio are set to balance model complexity and avoid overfitting
cph = CoxPHFitter(penalizer=0.1, l1_ratio=0.5)
cph.fit(
    df_model[['time_adj', 'event'] + covariates],
    'time_adj',
    'event'
)

<lifelines.CoxPHFitter: fitted with 1904 total observations, 1103 right-censored observations>

In [31]:
cph.print_summary()

<lifelines.CoxPHFitter: fitted with 1904 total observations, 1103 right-censored observations>
             duration col = 'time_adj'
                event col = 'event'
                penalizer = 0.1
                 l1 ratio = 0.5
      baseline estimation = breslow
   number of observations = 1904
number of events observed = 801
   partial log-likelihood = -4880.47
         time fit was run = 2025-12-23 23:11:14 UTC

---
              coef exp(coef)  se(coef)  coef lower 95%  coef upper 95% exp(coef) lower 95% exp(coef) upper 95%
covariate                                                                                                     
brca1        -0.00      1.00      0.00           -0.00            0.00                1.00                1.00
brca2         0.00      1.00      0.00           -0.00            0.00                1.00                1.00
pten         -0.00      1.00      0.00           -0.00            0.00                1.00                1.00
tp53          0.00      1.00      0.00           -0.00            0.00                1.00                1.00
cdh1         -0.00      1.00      0.00           -0.00            0.00                1.00                1.00
chek2         0.00      1.00      0.00           -0.00            0.00                1.00                1.00
nbn           0.00      1.00      0.00           -0.00            0.00                1.00                1.00
bard1        -0.00      1.00      0.00           -0.00            0.00                1.00                1.00
mlh1          0.00      1.00      0.00           -0.00            0.00                1.00                1.00
msh2          0.00      1.00      0.00           -0.00            0.00                1.00                1.00
msh6          0.00      1.00      0.00           -0.00            0.00                1.00                1.00
pms2         -0.00      1.00      0.00           -0.00            0.00                1.00                1.00
epcam         0.00      1.00      0.00           -0.00            0.00                1.00                1.00
rad51c        0.00      1.00      0.00           -0.00            0.00                1.00                1.00
rad51d        0.00      1.00      0.00           -0.00            0.00                1.00                1.00
rad50         0.00      1.00      0.00           -0.00            0.00                1.00                1.00
rb1           0.03      1.03      0.04           -0.05            0.12                0.95                1.12
rbl1          0.00      1.00      0.00           -0.00            0.00                1.00                1.00
ccna1         0.00      1.00      0.00           -0.00            0.00                1.00                1.00
ccnb1        -0.00      1.00      0.00           -0.00            0.00                1.00                1.00
cdk1         -0.00      1.00      0.00           -0.00            0.00                1.00                1.00
ccne1         0.00      1.00      0.00           -0.00            0.00                1.00                1.00
cdc25a        0.00      1.00      0.00           -0.00            0.00                1.00                1.00
ccnd1        -0.00      1.00      0.00           -0.00            0.00                1.00                1.00
cdk4          0.00      1.00      0.00           -0.00            0.00                1.00                1.00
cdk6          0.00      1.00      0.00           -0.00            0.00                1.00                1.00
ccnd2         0.00      1.00      0.00           -0.00            0.00                1.00                1.00
cdkn2a        0.00      1.00      0.00           -0.00            0.00                1.00                1.00
cdkn2b        0.00      1.00      0.00           -0.00            0.00                1.00                1.00
myc           0.05      1.05      0.04           -0.03            0.13                0.97                1.14
cdkn1b        0.00 

### Model Result
Many genes given coefficient of 0 due to L1 regularization.

### Next steps: 
1. Tune penalizer via k-fold cross validation with an array of penalizer values

In [32]:
from lifelines.utils import k_fold_cross_validation

for p in [0.001, 0.01, 0.05, 0.1]:
    cph = CoxPHFitter(penalizer=p, l1_ratio=0.5)
    scores = k_fold_cross_validation(
        cph,
        df_model[['time_adj', 'event'] + covariates],
        duration_col='time_adj',
        event_col='event',
        k=5,
        scoring_method="concordance_index"
    )
    print(p, np.mean(scores))


0.001 0.667591670559224
0.01 0.680085385054688


/Users/jmakings/miniconda3/envs/bc_survival/lib/python3.11/site-packages/scipy/_lib/_util.py:1226: LinAlgWarning: Ill-conditioned matrix (rcond=1.04604e-16): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/Users/jmakings/miniconda3/envs/bc_survival/lib/python3.11/site-packages/scipy/_lib/_util.py:1226: LinAlgWarning: Ill-conditioned matrix (rcond=7.99568e-17): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/Users/jmakings/miniconda3/envs/bc_survival/lib/python3.11/site-packages/scipy/_lib/_util.py:1226: LinAlgWarning: Ill-conditioned matrix (rcond=5.10617e-17): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/Users/jmakings/miniconda3/envs/bc_survival/lib/python3.11/site-packages/scipy/_lib/_util.py:1226: LinAlgWarning: Ill-conditioned matrix (rcond=3.92898e-17): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/Users/jmakings/miniconda3/envs/bc_survival/lib/python3.11/site-packages/scipy/_lib/

0.05 0.7025629447900245
0.1 0.6856797684341873
